# Track A — genuine full-duplex Urdu via Moshi LoRA

**The ambitious track.** Moshi is natively full-duplex: it listens and speaks
*at the same time* over a stereo stream (caller on one channel, bot on the
other). No cascade, no barge-in hack. The plan, straight from the J-Moshi
recipe:

1. Build a two-party **stereo** Urdu corpus (no such thing exists — we
   synthesize it).
2. **Swap the tokenizer**: replace Moshi's English SentencePiece with an Urdu
   one. This is the single highest-leverage change.
3. **LoRA fine-tune** (rank 128) on a rented A100.

**Honest framing.** This is a *proof of path*, not a fluent product. And it
needs **~24 GB of VRAM** — it does **not** fit a free Kaggle T4 (16 GB). Steps
1, 2 and the config in this notebook run on **CPU**; only the LoRA run itself
needs the GPU, and you rent that.

## 1. Setup

Install with the `[moshi]` extra — that pulls SentencePiece, the only piece
the tokenizer swap needs. Nothing here imports torch or moshi; the GPU comes
later.

In [ ]:
# >>> run once.
import os, subprocess, sys
REPO = os.environ.get('DUPLEX_BOL_REPO', '/kaggle/working/duplex-bol')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/maidah-binte-tariq/duplex-bol', REPO], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[moshi]'], check=True)

import duplex_bol
import sentencepiece  # proves the [moshi] extra installed
print('ready: duplex_bol + sentencepiece')

## 2. Synthetic two-party stereo corpus (runs on CPU)

The load-bearing trick. Moshi learns from stereo where speaker A is on the left
channel and speaker B on the right, overlaps and all. We fabricate that: take
single-speaker clips, assign two speakers to two channels, lay them on a shared
timeline with a little **deliberate overlap** so the model sees simultaneous
speech — the thing that makes full-duplex full-duplex.

Here we use synthetic tones (distinct frequencies = distinct voices) so the cell
actually produces a real stereo WAV + manifest with bytes on disk. Mirrors
`scripts/make_demo_corpus.py`. In a real run, swap the tones for the Urdu TTS
clips of your selected speakers.

In [ ]:
import json
from pathlib import Path
import numpy as np
from duplex_bol.audio import write_wav
from duplex_bol.data import (
    DialogueConfig, SpeakerClip, build_dialogue, validate_manifest, write_jsonl,
)

OUT = Path('data/demo')
SR = 24_000  # Moshi's sample rate

def tone(freq, seconds=1.2, sr=SR):
    """A faded sine — stands in for one speaker's voice."""
    t = np.arange(int(seconds * sr)) / sr
    env = np.minimum(1.0, np.minimum(t * 10, (seconds - t) * 10))  # no click at edges
    return (0.4 * env * np.sin(2 * np.pi * freq * t)).astype(np.float32)

agent_lines = ['السلام علیکم', 'میں آپ کی کیا مدد کر سکتا ہوں', 'بہت شکریہ']
user_lines  = ['وعلیکم السلام', 'مجھے اپنا بیلنس جاننا ہے', 'ٹھیک ہے']

agent = [SpeakerClip('spk_agent', tone(180.0), txt, SR) for txt in agent_lines]
user  = [SpeakerClip('spk_user',  tone(240.0), txt, SR) for txt in user_lines]

# turn order: agent, user, agent, user, ...  (clip 0 is spoken first)
turn_order = [agent[0], user[0], agent[1], user[1], agent[2], user[2]]

# overlap_s > 0 starts each turn slightly before the last finishes -> the
# channels overlap in time. Keep it small (<= ~0.4 s) or it's a shouting match.
stereo, dialogue = build_dialogue(turn_order, DialogueConfig(overlap_s=0.15))
print('stereo shape', stereo.shape, '(samples, 2 channels)')
print(f'duration {dialogue.duration_s:.2f}s, {len(dialogue.turns)} turns, agent=left / user=right')

In [ ]:
# Write the WAV, then fill the audio_path the synthesizer left blank, then the
# manifest + the toolkit's index ({path, duration} rows, not pydantic models).
stereo_rel = 'trackA/call_000.wav'
write_wav(OUT / stereo_rel, stereo, dialogue.sample_rate)
dialogue.audio_path = stereo_rel
write_jsonl(OUT / 'trackA/dialogues.jsonl', [dialogue])

from duplex_bol.moshi import build_index
index_path = OUT / 'trackA/index.jsonl'
with index_path.open('w', encoding='utf-8') as fh:
    for row in build_index([dialogue]):
        fh.write(json.dumps(row, ensure_ascii=False) + '\n')
print('wrote', stereo_rel, '+ dialogues.jsonl + index.jsonl')

In [ ]:
# Validate BEFORE any GPU: checks the WAV exists, turns stay inside the clip,
# and no same-channel overlap (cross-channel overlap is allowed — it's the
# simultaneous speech Moshi learns from). Empty list == clean.
problems = validate_manifest([dialogue], audio_root=OUT)
print('manifest problems:', problems or '(none — clean)')

## 3. The tokenizer swap (the key step, runs on CPU)

Why this matters most: Moshi reads text through an **English** tokenizer. Feed
it Nastaliq and it shatters every word into byte-fallback fragments, so the
model never sees coherent Urdu sub-words. J-Moshi's win for Japanese came mostly
from swapping this piece. We do the same for Urdu — and normalize the corpus
first (the same normalizer the eval uses) so the tokenizer learns one canonical
spelling instead of three.

In [ ]:
from duplex_bol.moshi import prepare_corpus, train_urdu_tokenizer, UrduTokenizer

# In a real run, feed EVERY transcript you have. Here: the dialogue lines.
all_texts = agent_lines + user_lines
corpus_path = OUT / 'trackA/urdu_corpus.txt'
n = prepare_corpus(all_texts, corpus_path)  # normalizes, one line per sentence
print(f'{n} normalized lines -> {corpus_path}')

In [ ]:
# Train the SentencePiece model.
# NOTE: this toy corpus is a handful of lines, so we turn OFF byte_fallback
# (it adds 256 byte pieces) and OFF hard_vocab_limit, and ask for a small vocab.
# SentencePiece will train to whatever vocab the corpus actually supports.
# A REAL run keeps both ON and uses vocab_size ~8000 (the J-Moshi-style default).
model_prefix = OUT / 'trackA/urdu_tokenizer/urdu'
model_path = train_urdu_tokenizer(
    corpus_path,
    model_prefix,
    vocab_size=200,        # real run: ~8000
    byte_fallback=False,   # real run: True
    hard_vocab_limit=False,  # real run: True
)
print('tokenizer model:', model_path)

In [ ]:
# Round-trip check: encode -> ids -> decode should return the original text.
tok = UrduTokenizer(model_path)
for s in ['السلام علیکم', 'بہت شکریہ']:
    ids = tok.encode(s)
    back = tok.decode(ids)
    print(f'{s!r:>30}  ->  {ids}  ->  {back!r}   {"OK" if back == s else "MISMATCH"}')
print('vocab size:', tok.vocab_size)

## 4. Moshi index + LoRA config (runs on CPU)

Everything the fine-tune needs that isn't the audio: a validated config and a
VRAM go/no-go. `estimate_vram_gb` is a guardrail to catch "this won't fit on a
24 GB card" *before* you rent one. `validate()` flags config mistakes (e.g. an
unset tokenizer path — the whole point of Track A).

In [ ]:
from duplex_bol.moshi import MoshiFinetuneConfig, estimate_vram_gb

cfg = MoshiFinetuneConfig(
    tokenizer_path=str(model_path),     # the swapped-in Urdu tokenizer
    data_index=str(index_path),
    # defaults mirror the moshi-finetune example: LoRA rank 128, grad checkpointing on.
)

print('estimated training VRAM ~', estimate_vram_gb(cfg), 'GB')
print('config problems:', cfg.validate() or '(none)')

In [ ]:
# Write the YAML the toolkit reads (round-trips via MoshiFinetuneConfig.from_yaml).
yaml_path = OUT / 'trackA/moshi_lora.yaml'
print(cfg.to_yaml(yaml_path))
print('wrote', yaml_path)

> **Reality check.** The estimate above is ~16 GB+ for the frozen bf16 weights
> alone, and real activation memory pushes a live run to **~24 GB**. A free
> Kaggle T4 has 16 GB — not enough. Rent an A100 (40/80 GB) for the next cell.

## 5. The LoRA run (GPU / A100 only)

This is the only GPU step. We hand the **Urdu tokenizer** and the **stereo
index** to the official toolkit and run LoRA at rank 128.

- Toolkit: https://github.com/kyutai-labs/moshi-finetune
- Multilingual fork (the Urdu-relevant recipe):
  https://github.com/nu-dialogue/moshi-finetune

**Needs ~24 GB VRAM.** Will OOM on a free T4.

In [ ]:
# >>> needs GPU (A100). Reference commands — do not expect this to run on free Kaggle.
#
# # 1. get the toolkit
# !git clone https://github.com/kyutai-labs/moshi-finetune
# %cd moshi-finetune
# !pip install -e .
#
# # 2. point its config at OUR index + OUR Urdu tokenizer.
# #    duplex_bol already wrote a compatible config above (moshi_lora.yaml):
# #      base_model:     kyutai/moshiko-pytorch-bf16   (CC-BY-4.0)
# #      tokenizer_path: .../urdu_tokenizer/urdu.model  (our vocab swap)
# #      data_index:     .../trackA/index.jsonl
# #      lora.rank:      128
# #    Map those fields onto the toolkit's expected config keys, then:
#
# !python -m moshi_finetune.train --config /path/to/moshi_lora.yaml
#
# # 3. after training, run inference on a held-out Urdu stereo call and listen
# #    for: does it take turns? does it handle overlap without freezing?
print('LoRA training is A100-only; this cell documents the commands.')

## 6. Licensing + next steps

**Read this before shipping anything.**

| Asset | License | Commercial use? |
|-------|---------|-----------------|
| Moshi weights (`kyutai/moshiko-pytorch-bf16`) | CC-BY-4.0 | Yes (with attribution) |
| J-Moshi *released* weights | CC-BY-**NC** | **No** |
| Mozilla 3-speaker Urdu set | CC-BY-**NC** | **No** |
| Common Voice Urdu | CC0 | Yes |

So: the **base Moshi weights are fine commercially**, but if you train on the
Mozilla 3-speaker set (or start from J-Moshi's released weights) your result
inherits a **non-commercial** restriction. For a commercial product, train on
CC0 Common Voice (plus your own licensed recordings) instead.

**Next steps**

- Replace the synthetic tones with real Urdu TTS clips of distinct speakers, so
  the stereo corpus carries real phonetics, not sine waves.
- Scale the tokenizer corpus to thousands of lines; train at vocab ~8000 with
  `byte_fallback=True`.
- Run the rank-128 LoRA on an A100; evaluate turn-taking + overlap on held-out
  calls.
- Compare against Track B (Notebook 1): the cascade is the safe deliverable;
  this is the path to *genuine* simultaneity.